In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import svm
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay , accuracy_score
import matplotlib.pyplot as plt
import os

# Wisconsin Dataset

In [4]:
wisconsin=pd.read_csv("D:\Machine Learning\Datasets\Cases\Wisconsin\BreastCancer.csv",index_col=0)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
wisconsin['Class'] = le.fit_transform(wisconsin['Class'])
X,y=wisconsin.drop('Class',axis=1),wisconsin['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25,stratify=y)
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
wisconsin

Training samples: 489, Test samples: 210


,Clump,UniCell_Size,Uni_CellShape,MargAdh,SEpith,BareN,BChromatin,NoemN,Mitoses,Class
Code,,,,,,,,,,
61634,5,4,3,1,2,2,2,3,1,0
63375,9,1,2,6,4,10,7,7,2,1
76389,10,4,7,2,2,8,6,1,1,1
95719,6,10,10,10,8,10,7,10,7,1
128059,1,1,1,1,2,5,5,1,1,0
...,...,...,...,...,...,...,...,...,...,...
1369821,10,10,10,10,5,10,10,10,7,1
1371026,5,10,10,10,4,10,5,6,3,1
1371920,5,1,1,1,2,1,3,2,1,0


# Without Scaling

In [5]:
from sklearn.svm import SVC


svm=SVC(kernel="linear")
svm.fit(X_train,y_train)
y_pred=svm.predict(X_test)
print(accuracy_score(y_test,y_pred))

0.9714285714285714


# Without Scaling

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC  # or SVR for regression

# Create pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', C=1.0, gamma='scale'))
])

# Fit the pipeline (scaling and training happen together)
pipeline.fit(X_train, y_train)

# Make predictions (scaling is applied automatically)
y_pred = pipeline.predict(X_test)

# Evaluate

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')


Accuracy: 0.9666666666666667


# HR Dataset


"D:\Machine Learning\Datasets\HR_comma_sep.csv"

In [7]:
hr=pd.read_csv("D:\Machine Learning\Datasets\HR_comma_sep.csv")
hr

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [8]:
X , y = hr.drop('left',axis=1) , hr['left']
ohe = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")
col_transformer = ColumnTransformer([("OHE", ohe, make_column_selector(dtype_include=object))], 
                                    remainder='passthrough', 
                                    verbose_feature_names_out=False)
col_transformer = col_transformer.set_output(transform="pandas")
X = col_transformer.fit_transform(X)
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.3,random_state=25,stratify=y)

In [9]:
std_scaler=StandardScaler()
svm=SVC(kernel="linear")


# With Scaling
 

In [10]:

pipe=Pipeline([("TRNSF",col_transformer),("SCL",std_scaler),("SVC",svm)])
pipe.fit(X_train,y_train)
y_pred=pipe.predict(X_test)
print(accuracy_score(y_test,y_pred))

0.7819515447877307


In [12]:
cs=np.linspace(.001,5,15)
cs

array([1.00000000e-03, 3.58071429e-01, 7.15142857e-01, 1.07221429e+00,
       1.42928571e+00, 1.78635714e+00, 2.14342857e+00, 2.50050000e+00,
       2.85757143e+00, 3.21464286e+00, 3.57171429e+00, 3.92878571e+00,
       4.28585714e+00, 4.64292857e+00, 5.00000000e+00])

In [16]:
cs=np.linspace(.001,5,15)
scores=[]
for c in cs:
    svm=SVC(kernel="linear",C=c)
    pipe=Pipeline([("TRNSF",col_transformer),("SCL",std_scaler),("SVC",svm)])
    pipe.fit(X_train,y_train)
    y_pred=pipe.predict(X_test)
    scores.append([c,accuracy_score(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=('c','score'))
df_scores.sort_values('score',ascending=False)

,c,score
1,0.358071,0.781952
2,0.715143,0.781952
3,1.072214,0.781952
5,1.786357,0.781952
4,1.429286,0.781952
6,2.143429,0.781952
7,2.500500,0.781952
11,3.928786,0.781952
8,2.857571,0.781952
9,3.214643,0.781952


# Polynomial Kernal

In [17]:
std_scaler=StandardScaler()
svm=SVC(kernel="poly")

In [18]:

pipe=Pipeline([("TRNSF",col_transformer),("SCL",std_scaler),("SVC",svm)])
pipe.fit(X_train,y_train)
y_pred=pipe.predict(X_test)
print(accuracy_score(y_test,y_pred))

0.9415425650144477


# Parameter Tuning for Polynomial kernal

In [24]:
cs=np.linspace(.001,5,15)
scores=[]
for c in cs:
    svm=SVC(kernel="poly",C=c)
    pipe=Pipeline([("TRNSF",col_transformer),("SCL",std_scaler),("SVC",svm)])
    pipe.fit(X_train,y_train)
    y_pred=pipe.predict(X_test)
    scores.append([c,accuracy_score(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=('c','score'))
df_scores.sort_values('score',ascending=False)

,c,score
13,4.642929,0.952212
11,3.928786,0.951767
14,5.000000,0.951767
12,4.285857,0.951545
10,3.571714,0.950878
8,2.857571,0.950433
9,3.214643,0.949989
7,2.500500,0.949767
6,2.143429,0.949100
5,1.786357,0.947766


In [29]:
cs=np.linspace(.001,5,15)
degree=[2,3,4]
scores=[]
for d in tqdm(degree):
    for c in cs:
        svm=SVC(kernel="poly",C=c,degree=d)
        pipe=Pipeline([("TRNSF",col_transformer),("SCL",std_scaler),("SVC",svm)])
        pipe.fit(X_train,y_train)
        y_pred=pipe.predict(X_test)
        scores.append([c,d,accuracy_score(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=('c','d','score'))
df_scores.sort_values('score',ascending=False)

100%|██████████| 3/3 [01:01<00:00, 20.44s/it]


,c,d,score
41,3.928786,4,0.956879
40,3.571714,4,0.956657
39,3.214643,4,0.956435
42,4.285857,4,0.956212
44,5.000000,4,0.955990
38,2.857571,4,0.955768
43,4.642929,4,0.955768
37,2.500500,4,0.955101
36,2.143429,4,0.953990
35,1.786357,4,0.953545


# Radial Kernal

In [25]:
std_scaler=StandardScaler()
svm=SVC(kernel="rbf")

In [26]:

pipe=Pipeline([("TRNSF",col_transformer),("SCL",std_scaler),("SVC",svm)])
pipe.fit(X_train,y_train)
y_pred=pipe.predict(X_test)
print(accuracy_score(y_test,y_pred))

0.9524338741942654


# Parameter Tuning for Radial kernal

In [27]:
cs=np.linspace(.001,5,15)
scores=[]
for c in cs:
    svm=SVC(kernel="rbf",C=c)
    pipe=Pipeline([("TRNSF",col_transformer),("SCL",std_scaler),("SVC",svm)])
    pipe.fit(X_train,y_train)
    y_pred=pipe.predict(X_test)
    scores.append([c,accuracy_score(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=('c','score'))
df_scores.sort_values('score',ascending=False)

,c,score
14,5.000000,0.957768
13,4.642929,0.957324
12,4.285857,0.956879
11,3.928786,0.956435
9,3.214643,0.955768
10,3.571714,0.955768
8,2.857571,0.954879
7,2.500500,0.954657
6,2.143429,0.954212
5,1.786357,0.953768


In [33]:
gammas=cs=np.linspace(.001,5,5)
cs=np.linspace(.001,5,15)
scores=[]
for gamma in tqdm(gammas):
    for c in tqdm(cs):
        svm=SVC(kernel="rbf",C=c,gamma=gamma)
        pipe=Pipeline([("TRNSF",col_transformer),("SCL",std_scaler),("SVC",svm)])
        pipe.fit(X_train,y_train)
        y_pred=pipe.predict(X_test)
        scores.append([c,d,accuracy_score(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=('c','gamma','score'))
df_scores.sort_values('score',ascending=False)

100%|██████████| 5/5 [09:13<00:00, 110.63s/it]


,c,gamma,score
20,1.786357,2,0.982218
19,1.429286,2,0.981107
21,2.143429,2,0.980662
28,4.642929,2,0.980440
29,5.000000,2,0.980440
...,...,...,...
2,0.715143,2,0.762169
30,0.001000,2,0.762169
15,0.001000,2,0.762169
45,0.001000,2,0.762169
